In [30]:
# import libraries (you may add additional imports but you may not have to)
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt

In [32]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/books/book-crossings.zip

!unzip -0 book-crossings.zip

books_filename = 'BX-Books.csv'
ratings_filename = 'BX-Book-Ratings.csv'

--2026-04-23 17:20:53--  https://cdn.freecodecamp.org/project-data/books/book-crossings.zip
Resolving cdn.freecodecamp.org (cdn.freecodecamp.org)... 104.26.3.33, 172.67.70.149, 104.26.2.33, ...
Connecting to cdn.freecodecamp.org (cdn.freecodecamp.org)|104.26.3.33|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 26085508 (25M) [application/zip]
Saving to: ‘book-crossings.zip.6’

book-crossings.zip. 100%[===================>]  24.88M   156MB/s    in 0.2s    

2026-04-23 17:20:54 (156 MB/s) - ‘book-crossings.zip.6’ saved [26085508/26085508]

UnZip 6.00 of 20 April 2009, by Debian. Original by Info-ZIP.

Usage: unzip [-Z] [-opts[modifiers]] file[.zip] [list] [-x xlist] [-d exdir]
  Default action is to extract files in list, except those in xlist, to exdir;
  file[.zip] may be a wildcard.  -Z => ZipInfo mode ("unzip -Z" for usage).

  -p  extract files to pipe, no messages     -l  list files (short format)
  -f  freshen existing files, create none    -t  test compr

In [33]:
# 1. Reload the data to ensure we aren't working with empty variables
df_books = pd.read_csv(
    books_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['isbn', 'title', 'author'],
    usecols=['isbn', 'title', 'author'],
    dtype={'isbn': 'str', 'title': 'str', 'author': 'str'})

df_ratings = pd.read_csv(
    ratings_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['user', 'isbn', 'rating'],
    usecols=['user', 'isbn', 'rating'],
    dtype={'user': 'int32', 'isbn': 'str', 'rating': 'float32'})

# 2. Filter out users with < 200 ratings
user_counts = df_ratings['user'].value_counts()
df_ratings_filtered = df_ratings[df_ratings['user'].isin(user_counts[user_counts >= 200].index)]

# 3. Filter out books with < 100 ratings
book_counts = df_ratings_filtered['isbn'].value_counts()
df_ratings_filtered = df_ratings_filtered[df_ratings_filtered['isbn'].isin(book_counts[book_counts >= 100].index)]

# 4. Merge and Pivot
df_merged = pd.merge(df_ratings_filtered, df_books, on='isbn').drop_duplicates(['title', 'user'])
df_pivot = df_merged.pivot(index='title', columns='user', values='rating').fillna(0)

# 5. Fit the Model
matrix = csr_matrix(df_pivot.values)
model_knn = NearestNeighbors(metric='cosine', algorithm='brute')
model_knn.fit(matrix)

print("Setup complete! Data shape:", df_pivot.shape)


Setup complete! Data shape: (99, 857)


In [34]:
# Filter Users (>= 200) and Books (>= 100)
user_counts = df_ratings['user'].value_counts()
book_counts = df_ratings['isbn'].value_counts()

df_filtered = df_ratings[
    df_ratings['user'].isin(user_counts[user_counts >= 200].index) &
    df_ratings['isbn'].isin(book_counts[book_counts >= 100].index)
]

# Merge and Pivot
df_merged = pd.merge(df_filtered, df_books, on='isbn').drop_duplicates(['title', 'user'])
df_pivot = df_merged.pivot(index='title', columns='user', values='rating').fillna(0)

# Create Matrix and Fit Model
matrix = csr_matrix(df_pivot.values)
model_knn = NearestNeighbors(metric='cosine', algorithm='brute')
model_knn.fit(matrix)

print("Model trained successfully! Pivot shape:", df_pivot.shape)


Model trained successfully! Pivot shape: (673, 888)


In [36]:
def get_recommends(book = ""):
    # Get the distances and indices for the 6 closest books
    distances, indices = model_knn.kneighbors(df_pivot.loc[book].values.reshape(1, -1), n_neighbors=6)

    recommended_books = []
    # Loop backwards through indices 1-5 (excluding the book itself)
    for i in range(1, 6):
        recommended_books.append([df_pivot.index[indices[0][i]], distances[0][i]])

    # Reverse the list so the furthest is first, as the test expects
    return [book, recommended_books[::-1]]


In [37]:
books = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
print(books)

def test_book_recommendation():
  test_pass = True
  recommends = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
  if recommends[0] != "Where the Heart Is (Oprah's Book Club (Paperback))":
    test_pass = False
  recommended_books = ["I'll Be Seeing You", 'The Weight of Water', 'The Surgeon', 'I Know This Much Is True']
  recommended_books_dist = [0.8, 0.77, 0.77, 0.77]
  for i in range(2):
    if recommends[1][i][0] not in recommended_books:
      test_pass = False
    if abs(recommends[1][i][1] - recommended_books_dist[i]) >= 0.05:
      test_pass = False
  if test_pass:
    print("You passed the challenge! 🎉🎉🎉🎉🎉")
  else:
    print("You haven't passed yet. Keep trying!")

test_book_recommendation()

["Where the Heart Is (Oprah's Book Club (Paperback))", [["I'll Be Seeing You", np.float32(0.8016211)], ['The Weight of Water', np.float32(0.77085835)], ['The Surgeon', np.float32(0.7699411)], ['I Know This Much Is True', np.float32(0.7677075)], ['The Lovely Bones: A Novel', np.float32(0.7234864)]]]
You passed the challenge! 🎉🎉🎉🎉🎉


In [42]:
# 1. Reload raw data to clear any previous accidental filtering
df_books = pd.read_csv(books_filename, encoding="ISO-8859-1", sep=";", header=0, names=['isbn', 'title', 'author'], usecols=['isbn', 'title', 'author'], dtype={'isbn': 'str', 'title': 'str', 'author': 'str'})
df_ratings = pd.read_csv(ratings_filename, encoding="ISO-8859-1", sep=";", header=0, names=['user', 'isbn', 'rating'], usecols=['user', 'isbn', 'rating'], dtype={'user': 'int32', 'isbn': 'str', 'rating': 'float32'})

# 2. Identify users to keep (>= 200 ratings)
user_counts = df_ratings['user'].value_counts()
active_users = user_counts[user_counts >= 200].index

# 3. Identify books to keep (>= 100 ratings)
book_counts = df_ratings['isbn'].value_counts()
popular_books = book_counts[book_counts >= 100].index

# 4. Apply BOTH filters at the same time to the ORIGINAL data
df_filtered = df_ratings[df_ratings['user'].isin(active_users) & df_ratings['isbn'].isin(popular_books)]

# 5. Merge with books to get titles
df_final = pd.merge(df_filtered, df_books, on='isbn')

# 6. Pivot and handle duplicates
df_pivot = df_final.drop_duplicates(['title', 'user']).pivot(index='title', columns='user', values='rating').fillna(0)

# 7. Create matrix and model
matrix = csr_matrix(df_pivot.values)
model_knn = NearestNeighbors(metric='cosine', algorithm='brute')
model_knn.fit(matrix)

print(f"Success! Model trained with {df_pivot.shape[0]} books and {df_pivot.shape[1]} users.")


Success! Model trained with 673 books and 888 users.


In [39]:
model_knn = NearestNeighbors(metric='cosine', algorithm='brute')
model_knn.fit(matrix)

NearestNeighbors(algorithm='brute', metric='cosine')

In [40]:
# 1. Filter the data
user_counts = df_ratings['user'].value_counts()
df_ratings = df_ratings[df_ratings['user'].isin(user_counts[user_counts >= 200].index)]

book_counts = df_ratings['isbn'].value_counts()
df_ratings = df_ratings[df_ratings['isbn'].isin(book_counts[book_counts >= 100].index)]

# 2. Prepare the table
df_merged = pd.merge(df_ratings, df_books, on='isbn')
df_merged = df_merged.drop_duplicates(['title', 'user'])
df_pivot = df_merged.pivot(index='title', columns='user', values='rating').fillna(0)

# 3. Train the model
matrix = csr_matrix(df_pivot.values)
model_knn = NearestNeighbors(metric='cosine', algorithm='brute')
model_knn.fit(matrix)


NearestNeighbors(algorithm='brute', metric='cosine')